# PortBench: does Gemma 4 E2B depend on the order of the file list?

Code and data: https://github.com/zak01E/portbench (`gemma_shuffle.py`). The paths-only prompt is run three times per instance: original listing order (control) and two random orders of the same lines, on the same backend (4-bit NF4, greedy, thinking off, at most 350 new tokens). Scoring is done by `gemma_shuffle_analyze.py`. Needs the model `google/gemma-4/transformers/gemma-4-e2b-it` as input and a T4 GPU.

In [ ]:
!pip install -q -U transformers accelerate kagglehub bitsandbytes
!wget -q https://raw.githubusercontent.com/zak01E/portbench/master/data/llm_inputs.json
!wget -q https://raw.githubusercontent.com/zak01E/portbench/master/gemma_baseline.py
!wget -q https://raw.githubusercontent.com/zak01E/portbench/master/gemma_shuffle.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import kagglehub
path = kagglehub.model_download('google/gemma-4/transformers/gemma-4-e2b-it')
print(path)

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True GEMMA_OUT=/kaggle/working/gemma_e2b_shuffle.json python gemma_shuffle.py llm_inputs.json {path}

In [ ]:
import json
res = json.load(open('/kaggle/working/gemma_e2b_shuffle.json'))
print(len(res), 'instances; empty answers:', {n: sum(r[n + '_raw'] == '' for r in res) for n in ('listing', 'shuffle0', 'shuffle1')})